### Setup

In [ ]:
import pandas as pd
import numpy as np
import random

import torch
import torch.nn as nn
import torch.nn.functional as F

import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from common.utils import DataPreprocessor, FeatureEngineer, ExperimentDataPreprocessor, set_seed

MOVIELENS_DATA_DIR = "../datasets/hetrec2011-movielens-2k-v2/user_ratedmovies.dat"
RANDOM_SEED = 42
BATCH_SIZE = 1024
EMB_DIM = 64
LR = 1e-3
EPOCHS = 5

# Initialize data processors
set_seed(RANDOM_SEED)
data_preprocessor = DataPreprocessor()
feature_engineer = FeatureEngineer()
experiment_data_preprocessor = ExperimentDataPreprocessor()


/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
Seed set to 42


random seed set to 42
numpy seed set to 42
torch seed set to 42
lightning seed set to 42


### Load and Process DataFrame

In [2]:
interaction_df = data_preprocessor.load_and_process_df(
    file_dir=MOVIELENS_DATA_DIR,
    year_range=(2006, 2008),
)
interaction_df.head()

Data count: 855598
Data count after filtering by year (2006, 2008): 480608
Num of distinct users: 2103
Num of distinct items: 9519
done!
------------------------------
Filtering by min user/item interactions (0/0):
Data count before: 480608
Data count after: 480608
done!
------------------------------
==== Final Data Info: ====
Data Year Range: (2006, 2008)
Rating Threshold: 4.0
Num of interactions: 480608
Num of distinct users: 2103
Num of distinct items: 9519


,userID,movieID,rating,date_day,date_month,date_year,date_hour,date_minute,date_second,timestamp,label
0,75,3,1.0,29,10,2006,23,17,16,2006-10-29 23:17:16,0
1,75,32,4.5,29,10,2006,23,23,44,2006-10-29 23:23:44,1
2,75,110,4.0,29,10,2006,23,30,8,2006-10-29 23:30:08,1
3,75,160,2.0,29,10,2006,23,16,52,2006-10-29 23:16:52,0
4,75,163,4.0,29,10,2006,23,29,30,2006-10-29 23:29:30,1


### Join Side Information

In [3]:
interaction_info_df = data_preprocessor.join_item_features(
    df=interaction_df, actor_k=5,
)
interaction_info_df.head()

extracting item features...
merging features...
interaction data count before merging: 480608
interaction data count after merging: 478564
done!


,userID,movieID,rating,date_day,date_month,date_year,date_hour,date_minute,date_second,timestamp,label,actorID,country,directorID,directorName,genre
0,75,3,1.0,29,10,2006,23,17,16,2006-10-29 23:17:16,0,"[jack_lemmon, walter_matthau, annmargret, burg...",USA,donald_petrie,Donald Petrie,"[Comedy, Romance, [PAD], [PAD], [PAD], [PAD], ..."
1,75,32,4.5,29,10,2006,23,23,44,2006-10-29 23:23:44,1,"[bhiravi_vaidhy, dilip_satgare, haresh_mehta, ...",USA,siddharth_randeria,Siddharth Randeria,"[Sci-Fi, Thriller, [PAD], [PAD], [PAD], [PAD],..."
2,75,110,4.0,29,10,2006,23,30,8,2006-10-29 23:30:08,1,"[mel_gibson, sophie_marceau, patrick_mcgoohan,...",USA,mel_gibson,Mel Gibson,"[Action, Drama, War, [PAD], [PAD], [PAD], [PAD..."
3,75,160,2.0,29,10,2006,23,16,52,2006-10-29 23:16:52,0,"[dylan_walsh, laura_linney, ernie_hudson_jr, t...",USA,frank_marshall,Frank Marshall,"[Action, Adventure, Mystery, Sci-Fi, [PAD], [P..."
4,75,163,4.0,29,10,2006,23,29,30,2006-10-29 23:29:30,1,"[antonio_banderas, salma_hayek, 1142520-joaqui...",USA,robert_rodriguez,Robert Rodriguez,"[Action, Romance, Thriller, [PAD], [PAD], [PAD..."


### Prepare Train/Valid/Test Set

In [4]:
# TODO: determine which method to use for splitting
# 1. Split by year
# 2. Stratified split by user, timestamp

train_df, valid_df, test_df = experiment_data_preprocessor.stratified_time_split(
    interaction_info_df,
    time_col="timestamp",
    train_ratio=0.75,
    val_ratio=0.1,
    test_ratio=0.15,
)

TRAIN_NUM_USERS = len(train_df["userID"].unique())
TRAIN_NUM_ITEMS = len(train_df["movieID"].unique())


Splitting data into train/valid/test by time period with ratio=(0.75 : 0.1 : 0.15):
train: 358129 (74.83%
valid: 46916 (9.8%)
test: 73519 (15.36%)
------------------------------ 

Check target label distribution after splitting (%):
train label
0    0.555886
1    0.444114
Name: proportion, dtype: float64
valid label
0    0.610772
1    0.389228
Name: proportion, dtype: float64
test label
0    0.585155
1    0.414845
Name: proportion, dtype: float64


### Re-index User/Item ID & Encode Categorical Features

In [5]:
print("Train: fit_transform")
encoded_train_df = feature_engineer.fit_transform(train_df)
print("---"*10)
print("Valid: transform")
encoded_valid_df = feature_engineer.transform(valid_df)
print("---"*10)
print("Test: transform")
encoded_test_df = feature_engineer.transform(test_df)
print("---"*10)

Train: fit_transform
Re-index mapping dumped into ...
user: ../datasets/userid_mapping.csv
item: ../datasets/itemid_mapping.csv
Fitted: user/item mapping
Fitted: vocab2idx for actorID
Fitted: vocab2idx for country
Fitted: vocab2idx for directorID
Fitted: vocab2idx for genre
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
------------------------------
Valid: transform
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
------------------------------
Test: transform
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
------------------------------


In [6]:
# # NOTE: can check the encoding vocab idx content from the feature engineer
# oov_idx = feature_engineer.vocab2idx["movieID"]["[OOV]"]
# len(test_df[test_df["movieID"] == oov_idx])

### Prepare Additional Data for Train/Inference

#### Build bi-partite graph for training

In [7]:
# NOTE: At training, we use interaction graph from train_df for train and validation
train_graph = experiment_data_preprocessor.create_interaction_graph(encoded_train_df)

# NOTE: At inference, we can use graph of (train_df + valid_df)
# train_valid_graph = utils.create_interaction_graph(pd.concat([train_df, valid_df], axis=0))

Creating interaction graph...
Drop negative samples
  Num of all interactions: 358129
  Num of positive interactions: 159050 

Building edges...
Building bi-directed edges (duplications)...
Building labels...
Building bi-directed labels (duplications)... 

Interaction Graph: Data(edge_index=[2, 318100], edge_label=[318100])
Edge Index: tensor([[   0,    0,    0,  ..., 2835,  742, 2969],
        [1102, 1186,  670,  ..., 2093, 2093, 2093]])


#### Prepare prediction pool for inference/testing

In [8]:
# NOTE: Prepare prediction pool to evaluate the model
prediction_pool_df = experiment_data_preprocessor.prepare_prediction_df(encoded_test_df, K=500)
prediction_pool_df.tail()

Prediction DataFrame:
User Pool: 2095
Item Pool: 6959, negative sampled to 500 items for each user
Num of interactions: 2095(users) * 500(items) = 1047500


,userID,movieID,label,actorID_idx,country_idx,directorID_idx,genre_idx
1047495,2094,7957,0,"[5403, 102, 5844, 14185, 8714]",63,955,"[8, 15, 0, 0, 0, 0, 0, 0]"
1047496,2094,1611,0,"[13198, 4752, 15371, 5573, 8531]",63,2910,"[4, 5, 0, 0, 0, 0, 0, 0]"
1047497,2094,4715,0,"[2653, 2800, 5578, 9181, 11244]",63,1393,"[7, 18, 0, 0, 0, 0, 0, 0]"
1047498,2094,1924,0,"[3069, 7855, 11716, 8412, 5873]",62,2293,"[5, 0, 0, 0, 0, 0, 0, 0]"
1047499,2094,1590,0,"[6359, 5740, 13573, 3459, 14058]",63,2954,"[1, 2, 8, 0, 0, 0, 0, 0]"


### Prepare DataLoader

In [9]:
# TODO: determine which Dataset to use
from common.datasets import UserItemPairDataset
from torch.utils.data import DataLoader

train_dataset = UserItemPairDataset(encoded_train_df)
valid_dataset = UserItemPairDataset(encoded_valid_df)
test_dataset = UserItemPairDataset(prediction_pool_df)
print("train data count:", len(train_dataset))
print("valid data count:", len(valid_dataset))
print("test data count:", len(test_dataset))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)


train data count: 358129
valid data count: 46916
test data count: 1047500


### Configure Model (LightningModule)

In [10]:
from models.gcn_rec import GCNRecID

model = GCNRecID(
    edge_index=train_graph.edge_index,  # shape [2, num_edges]
    num_user=TRAIN_NUM_USERS,
    num_item=TRAIN_NUM_ITEMS,
    dim_id=EMB_DIM,
    lr=LR,
)

### Configure Trainer and Experiment

In [11]:
from common._mlflow import get_mlflow_logger, get_callbacks

EXPERIMENT_NAME = "gcn-bce-exp"
RUN_NAME = "gcn-baseline-test0"
mlflow_logger = get_mlflow_logger(experiment_name=EXPERIMENT_NAME, run_name=RUN_NAME)
trainer_callbacks = get_callbacks(RUN_NAME, patience=3)

In [12]:
from pytorch_lightning import Trainer

trainer = Trainer(
    max_epochs=EPOCHS,
    logger=mlflow_logger,
    log_every_n_steps=1,
    callbacks=trainer_callbacks,
    accelerator='auto',  # or 'auto', 'gpu'
    # devices=[0], # if gpu is available
)


Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


### Train Model

In [13]:
# Start training
trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=valid_loader)


You are using a CUDA device ('NVIDIA GeForce RTX 4070 SUPER') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision


Experiment with name gcn-bce-exp not found. Creating it.
/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:654: Checkpoint directory /media/emma/10TB/home/bilab_archive/Bai/DPRecSys/basic/test_checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name  | Type          | Params | Mode 
------------------------------------------------
0 | model | GCN_ID_MODULE | 740 K  | train
------------------------------------------------
740 K     Trainable params
0         Non-trainable params
740 K     Total params
2.963     Total estimated model params size (MB)
13        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


🏃 View run gcn-baseline-test0 at: http://140.112.106.216:3683/#/experiments/3/runs/a7ff4b052e75405181b5cac498c9fce1
🧪 View experiment at: http://140.112.106.216:3683/#/experiments/3


ValueError: too many values to unpack (expected 3)

### Inference

In [11]:
# NOTE: the inference model MUST be the same as the training model
best_model_path = "test_checkpoints/gcn-baseline-test11-best-checkpoint-epoch=23-val_f1=0.65.ckpt"
model = GCNRecID.load_from_checkpoint(
    checkpoint_path=best_model_path, 
    edge_index=train_graph.edge_index,
    num_user=NUM_USER,
    num_item=NUM_ITEM,
    dim_id=EMB_DIM,
    lr=LR,
)


/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [12]:
# start inference
trainer.test(model=model, dataloaders=test_loader)

/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Testing: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.5716319680213928     │
│          test_f1          │    0.08631569892168045    │
│         test_loss         │    2.4332122802734375     │
│         test_prec         │   0.046012409031391144    │
│         test_rec          │    0.6956578493118286     │
└───────────────────────────┴───────────────────────────┘

🏃 View run gcn-baseline-test12 at: http://140.112.106.216:3683/#/experiments/2/runs/f84bb291aa23446bbc154796205a30cf
🧪 View experiment at: http://140.112.106.216:3683/#/experiments/2


[{'test_loss': 2.4332122802734375,
  'test_acc': 0.5716319680213928,
  'test_prec': 0.046012409031391144,
  'test_rec': 0.6956578493118286,
  'test_f1': 0.08631569892168045}]